# 09 - Train a Supervised-Policy Model Offline

This notebook is the ranking check: can the model copy expert `argmax(Q*)`?

It uses the same data, embedder, and backbone as `05_train_offline_sv.ipynb`, but swaps the value head and `SvObjective` for `ClassificationHead` and `SpObjective`. The head is trained with hard cross-entropy onto a random argmax of `info_q_star` (ties broken uniformly) while the episode is running (`episode_done == 0`). There is no Bellman backup, no target network, and no magnitude regression.

1. Load previously collected `Datastore` streams from the Hub (must include `info_q_star`).
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` from an embedder, a backbone, and an action head.
4. Train with `SpObjective` and save with `push_model_to_hub`.

`Augmenter` remaps `action` ids and, via `input_vector_field` /
`output_vector_field` on `info_q_star`, reorders the Q vector with the **same**
permutation so it stays aligned with the remapped ids.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `15_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    to_device,
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import SpObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import TransformerBackbone
from mouse_core.models.heads import ClassificationHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-offline-sp"      # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-tokenizer-offline-sp"  # Hugging Face tokenizer repo (separate from MODEL_ID)
PRETRAINED = "Qwen/Qwen3-0.6B"                  # HF checkpoint for Tokenizer and backbone
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → backbone.embed`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` shares draws within one sampled sequence). Each window gets its own `reseed` generation, so the same index on two rollouts draws two seeds. Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `episode_done` / `info_q_star`; `grouping_field=`) |

Compose `train_transform = compose(stages=(augmenter, tokenizer))`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `15_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

GROUP_PREFIX = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. You have 20 episodes to solve the task. The "
    "grid is permuted, so squares are not in order; action ids may be "
    "remapped.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "Predict when you see a new line. Step format: "
    "action,observation[,r=reward][,d=done][,e=episode].\n"
)

# input_vector_field/output_vector_field share the action permute so Q*[perm[a]] stays aligned.

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "text",
            "input_field": "action",
            "format": "{field}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": ",r={field:g}",
            "skip": 0.0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "input_field": "episode_done",
            "format": ",d={field}",
            "skip": 0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "input_field": "episode_index",
            "format": ",e={field}",
            "when_field": "step_index",
            "when_equals": 0,
        },
        {
            "type": "text",
            "output_field": "value",
            "format": "\n",
            "max_tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "info_q_star",
        },
    ],
    grouping_field="task_index",
    group_prefix=GROUP_PREFIX,
    pretrained=PRETRAINED,
)

train_transform = compose(stages=(augmenter, tokenizer))

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has a backbone and heads:

- `TransformerBackbone(pretrained=...)` loads the checkpoint including `embed_tokens`, looks up the packed `__text__` ids, and runs the decoder. Step templates and field packing live on `Tokenizer` only.
- `ClassificationHead` predicts one logit per discrete action.

The backbone exposes `hidden_dim`, and the head uses that same value so the pieces connect cleanly.

`Tokenizer` text fields match `02`: comma-separated action / observation, `r=` / `d=` when nonzero, and a const newline readout flagged `head_output: True`.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache.


In [ ]:
backbone = TransformerBackbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
    pretrained=PRETRAINED,
)


head = ClassificationHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=1.0,
    use_norm=True,
)

model = Model(
    backbone=backbone,
    heads={"action": head},
    action_source="action",
    reasoner=None,
).train().to(device)
print(model)



## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`.

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone, and produces flat per-step head predictions.
3. `SpObjective` trains `predictions["action"]` with hard CE onto a uniformly random argmax of `objective_data["info_q_star"]` (ties are not biased toward the lowest action index).
4. `AdamW` updates weights. The backbone and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights. There is no Polyak / target network.

`SpObjective(mask_key="episode_done")` drops any step where `episode_done != 0` (terminated or truncated): the episode is over, so there is no next action to imitate. Attention is task-isolated by `task_index` so the backbone cannot mix maps.


In [ ]:
optimizer = AdamW(
    params=model.parameters(),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
objective = SpObjective(
    loss_type="ce",
    targets_key="info_q_star",
    mask_key="episode_done",
)

def run_train(*, model: Model, optimizer: AdamW, objective: SpObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        predictions = model(inputs).predictions["action"]
        loss, metrics = objective(objective_data=to_device(data=objective_data, device=device), predictions=predictions)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `15_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  sp={metrics['action']:.4f}")
loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to a different repo (`TOKENIZER_ID`). Later, `load_model` reconstructs the `Model` and `load_tokenizer` on the tokenizer repo reloads the packing spec — formats, skips, and `head_output` cannot be recovered from the embedder alone.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(model=model, tokenizer=tokenizer, repo_id=MODEL_ID, tokenizer_repo_id=TOKENIZER_ID, private=False, clear=True)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")